# Naive RAG vs CRAG on the same 50 HotpotQA questions

This tutorial compares a controlled naive-RAG baseline with the corrective pipeline described by Shi-Qi Yan, Jia-Chen Gu, Yun Zhu, and Zhen-Hua Ling in [*Corrective Retrieval Augmented Generation*, arXiv:2401.15884](https://arxiv.org/abs/2401.15884).

## Learning goals

By the end, you can identify what the comparison holds constant, separate string metrics from human answer quality, inspect paired failures, and judge whether correction helped on this 50-question slice.

## Fairness contract

Both methods receive the same question and identical ranked top-3 passages from the question's HotpotQA distractor pool. Both use `agnes-3.0-flash` and the same evidence-only answer generator. Naive RAG sends all complete passages directly to generation. CRAG evaluates, routes, selects exact sentence strips, and then generates. Web search is disabled. Gold answers and gold labels never enter model prompts.

The setup isolates the corrective stages. It does not reproduce the paper's trained T5-large evaluator or establish general accuracy. Human judgements cover all 50 pairs and are tied to exact answer/evidence fingerprints.

### Set up the reproducible environment

**Motivation.** Use the same repo paths, credentials, and fixed model as the main course.

**Paper mapping.** Infrastructure needed for the comparison.

**Next cell.** Resolve the repo root, import the comparison APIs, and verify credential presence without printing values.

**Failure signals.** Missing variables, a wrong kernel, or imports from another checkout stop execution before model calls.

**Read the output.** Both flags should be True and the model must be agnes-3.0-flash.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import check_environment, MODEL_NAME
from src.data_hotpot import build_slice
from src.qdrant_store import index_slice, close_qdrant_client
from src.retrieve import search
from src.pipeline import run_crag
from src.comparison import run_naive_rag, run_comparison_50, automated_metrics, load_manual_reviews

env = check_environment()
print(env)
assert all(env.values()) and MODEL_NAME == "agnes-3.0-flash"

{'AGNESAI_API_KEY': True, 'HF_TOKEN': True}


### Load the exact course slice and index

**Motivation.** A paired comparison needs the same questions and corpus for both methods.

**Paper mapping.** The paper motivates correction after retrieval. This adaptation uses HotpotQA.

**Next cell.** Load the seed-42 slice, index it idempotently, and lock the first 50 IDs.

**Failure signals.** A dataset provenance mismatch rebuilds the slice; a Qdrant lock means another kernel owns the store.

**Read the output.** Expect 200 records, 50 unique comparison IDs, and 1,992 indexed paragraphs.

In [2]:
records, _ = build_slice(n=200, seed=42)
indexed = index_slice(records)
comparison_ids = [row["id"] for row in records[:50]]
assert len(records) == 200 and len(set(comparison_ids)) == 50
print("Indexed paragraphs:", indexed)
print("Comparison IDs:", len(comparison_ids), comparison_ids[:3], "...", comparison_ids[-1])

[*] Collection 'hotpot_slice' already indexed with 1992 points. Skipping upsert.
Indexed paragraphs: 1992
Comparison IDs: 50 ['5add1d575542992c1e3a2540', '5ac55ea55542993e66e82377', '5ac3a76e554299741d48a2be'] ... 5ac0e064554299012d1db64e


## What “naive” means here

Naive RAG is retrieval → generation. It does not inspect relevance scores, choose Correct/Incorrect/Ambiguous, filter sentences, or search the web. It still uses a grounded answer prompt so both methods follow the same evidence rule. The baseline context is the full text of all three retrieved passages in rank order.

### Verify one shared retrieval set

**Motivation.** Check the experimental control before comparing answers.

**Paper mapping.** Both branches begin from the same retriever output; only the corrective stages differ.

**Next cell.** Retrieve the first question once and display its ranked titles, similarities, and diagnostic gold labels.

**Failure signals.** Different question IDs in payloads indicate a broken pool filter; similarity is not relevance confidence.

**Read the output.** The same three objects shown here are passed to both methods.

In [3]:
worked = records[0]
worked_hits = search(worked["question"], k=3, question_id=worked["id"])
assert all(hit["payload"]["question_id"] == worked["id"] for hit in worked_hits)
display(pd.DataFrame([{"rank": i + 1, "title": hit["title"], "similarity": hit["score"], "is_gold": hit["payload"]["is_gold"]} for i, hit in enumerate(worked_hits)]))

,rank,title,similarity,is_gold
0,1,Royal Flash (film),0.893053,True
1,2,Royal Flash,0.812065,False
2,3,Harry Flashman,0.801413,False


### Run naive RAG on complete passages

**Motivation.** Establish the baseline answer without evaluator or refinement calls.

**Paper mapping.** This is the uncorrected retrieval-to-generation path that CRAG is intended to improve.

**Next cell.** Pass the three complete retrieved texts directly to the grounded generator.

**Failure signals.** An abstention means the generator found the full retrieved context insufficient; it is not a transport failure.

**Read the output.** Inspect the answer and the exact titles supplied to it; gold labels were not included in the prompt.

In [4]:
worked_naive = run_naive_rag(worked["question"], worked_hits)
print("Question:", worked["question"])
print("Evidence titles:", worked_naive["evidence_titles"])
print("Naive answer:", worked_naive["answer"])
print("Actual requests:", worked_naive["n_llm_calls"])

Question: What nationality was Oliver Reed's character in the film Royal Flash?
Evidence titles: ['Royal Flash (film)', 'Royal Flash', 'Harry Flashman']
Naive answer: I don't have enough information in the provided evidence to answer this question.
Actual requests: 0


### Run CRAG on those identical passages

**Motivation.** Observe the effect of evaluation, routing, and strip refinement without changing retrieval.

**Paper mapping.** Maps directly to the paper's corrective control flow, implemented here with a prompted Agnes evaluator.

**Next cell.** Pass the already-retrieved hit objects into CRAG and display its decision and retained strips.

**Failure signals.** A Correct action does not prove both hops are present; an empty strip set leads to a safe abstention.

**Read the output.** Compare the retained evidence with the full naive context before comparing answer text.

In [5]:
worked_crag = run_crag(worked["question"], worked["id"], k=3, allow_web=False, docs=worked_hits)
print("Action:", worked_crag.action)
display(pd.DataFrame([{k: item[k] for k in ("title", "score", "label", "why")} for item in worked_crag.evaluations]))
print("Kept strips:", json.dumps(worked_crag.strips, indent=2, ensure_ascii=False))
print("CRAG answer:", worked_crag.answer)
print("Gold answer:", worked["gold_answer"])

Action: Correct


,title,score,label,why
0,Royal Flash (film),1.0,relevant,The evidence confirms Oliver Reed played Otto ...
1,Royal Flash,0.0,irrelevant,The evidence identifies the film and its sourc...
2,Harry Flashman,0.2,irrelevant,The evidence identifies the actor as Malcolm M...


Kept strips: [
  "Additionally, Oliver Reed appeared in the role of Otto von Bismarck, Alan Bates as Rudi von Sternberg, and Florinda Bolkan played Lola Montez."
]
CRAG answer: I don't have enough information in the provided evidence to answer this question.
Gold answer: Prussian


## Paired 50-question run

Each ID is atomically checkpointed, so a crash resumes at the failed ID. A first run can make one naive generation request plus CRAG evaluator, refinement, and generation requests per row; primitive CRAG caches can reduce that count. A warm rerun reports zero new calls. It does not erase the requests that created the cache.

### Execute all 50 paired questions

**Motivation.** Collect enough paired cases to see systematic wins, losses, abstentions, and metric disagreements.

**Paper mapping.** A tutorial comparison of the paper-inspired correction mechanism, not the paper benchmark.

**Next cell.** Run both methods on the same 50 questions with k=3 and web disabled.

**Failure signals.** Provider/schema failures are checkpointed and retried; exhausted retries stop without dropping the row.

**Read the output.** Every progress line identifies cache state and actual requests for that comparison invocation.

In [6]:
rows = run_comparison_50(records, k=3, allow_web=False)
assert len(rows) == 50 and [row["id"] for row in rows] == comparison_ids
print("Completed paired rows:", len(rows))

comparison: 1/50; cached=True; requests=0


comparison: 2/50; cached=True; requests=0


comparison: 3/50; cached=True; requests=0


comparison: 4/50; cached=True; requests=0


comparison: 5/50; cached=True; requests=0


...

comparison: 47/50; cached=True; requests=0


comparison: 48/50; cached=True; requests=0


comparison: 49/50; cached=True; requests=0


comparison: 50/50; cached=True; requests=0


Completed paired rows: 50


### Compare automated proxies

**Motivation.** Quantify retrieval coverage, string overlap, abstention, and current-run cost before human interpretation.

**Paper mapping.** These diagnostics are tutorial additions; they are not the paper's reported results.

**Next cell.** Compute paired metrics and a compact per-method table.

**Failure signals.** Substring match can reward negation or miss aliases; it must not be called accuracy.

**Read the output.** Look for different answer behavior despite identical title recall, then check whether this invocation was cold or cached.

In [7]:
auto = automated_metrics(rows)
print(json.dumps(auto, indent=2))
display(pd.DataFrame({
    "method": ["Naive RAG", "CRAG"],
    "substring_match": [auto["naive_substring_match"], auto["crag_substring_match"]],
    "abstention_rate": [auto["naive_abstention_rate"], auto["crag_abstention_rate"]],
    "actual_requests_this_run": [auto["naive_llm_calls"], auto["crag_llm_calls"]],
}))

{
  "n_questions": 50,
  "gold_title_recall@3": 0.42,
  "naive_substring_match": 0.3,
  "crag_substring_match": 0.26,
  "naive_abstention_rate": 0.66,
  "crag_abstention_rate": 0.72,
  "naive_llm_calls": 0,
  "crag_llm_calls": 0,
  "cached_rows": 50
}


,method,substring_match,abstention_rate,actual_requests_this_run
0,Naive RAG,0.30,0.66,0
1,CRAG,0.26,0.72,0


## Human audit of all 50 pairs

All 50 pairs were reviewed manually against the question, reference answer, and evidence given to each method. The audit is method-aware: the reviewer could see which answer came from naive RAG and which came from CRAG. That supports pipeline-specific error attribution and can introduce reviewer bias, which remains a limitation.

Correctness labels are `correct`, `partially_correct`, `incorrect`, and `abstain`. Evidence labels are `fully_supported`, `partially_supported`, `unsupported`, and `no_answer`. Pairwise preference weighs semantic correctness first, then evidence support, appropriate abstention, and concision or contradiction. A safe abstention is preferred to an unsupported wrong answer. Reviews are human judgements, not additional Agnes calls.

### Validate reviews against exact outputs

**Motivation.** Prevent a judgement from silently attaching to a regenerated answer or changed evidence set.

**Paper mapping.** Manual evaluation supplements automated metrics; it is not part of the CRAG algorithm.

**Next cell.** Load the tracked 50-row audit and compare its fingerprints with this run.

**Failure signals.** Missing or changed rows are listed for re-review while the notebook remains runnable.

**Read the output.** A fully reviewed run shows 50 valid and zero stale IDs.

In [8]:
review_path = ROOT / "reviews/naive_vs_crag_50_manual.json"
reviews, stale_ids = load_manual_reviews(rows, review_path)
print("Valid manual reviews:", len(reviews))
print("Stale or missing IDs:", stale_ids)

Valid manual reviews: 50
Stale or missing IDs: []


### Summarize human answer quality

**Motivation.** Use semantic and evidence judgements alongside the string heuristic.

**Paper mapping.** This is an external tutorial audit of the two pipelines. It is not a trained evaluator or paper metric.

**Next cell.** Count correctness, grounding, and pairwise preferences for all fingerprint-valid reviews.

**Failure signals.** If fewer than 50 reviews validate, totals are incomplete and no overall winner should be claimed.

**Read the output.** Compare method correctness with preference counts and note ties; the sample remains descriptive.

In [9]:
if len(reviews) == 50:
    manual_summary = {
        "naive_correctness": dict(Counter(r["naive_correctness"] for r in reviews)),
        "crag_correctness": dict(Counter(r["crag_correctness"] for r in reviews)),
        "naive_support": dict(Counter(r["naive_support"] for r in reviews)),
        "crag_support": dict(Counter(r["crag_support"] for r in reviews)),
        "preference": dict(Counter(r["preference"] for r in reviews)),
    }
    print(json.dumps(manual_summary, indent=2))
else:
    manual_summary = {}
    print("Manual summary withheld until all 50 exact outputs are reviewed.")

{
  "naive_correctness": {
    "abstain": 33,
    "correct": 15,
    "incorrect": 1,
    "partially_correct": 1
  },
  "crag_correctness": {
    "abstain": 36,
    "correct": 13,
    "incorrect": 1
  },
  "naive_support": {
    "no_answer": 33,
    "fully_supported": 15,
    "partially_supported": 2
  },
  "crag_support": {
    "no_answer": 36,
    "fully_supported": 12,
    "partially_supported": 2
  },
  "preference": {
    "tie_both_inadequate": 33,
    "tie_both_good": 11,
    "naive_better": 4,
    "crag_better": 2
  }
}


### Find string-metric disagreements

**Motivation.** Show why literal answer overlap cannot replace human inspection.

**Paper mapping.** The paper motivates stronger correction; this cell critiques our tutorial evaluation proxy.

**Next cell.** Join reviews to outputs and display cases where substring match disagrees with manual correctness.

**Failure signals.** An empty table means agreement on this run, not proof that substring matching is generally valid.

**Read the output.** Alias misses and incidental matches should be visible with their human rationale.

In [10]:
review_by_id = {review["id"]: review for review in reviews}
disagreements = []
for row in rows:
    review = review_by_id.get(row["id"])
    if not review:
        continue
    for method in ("naive", "crag"):
        manually_correct = review[f"{method}_correctness"] == "correct"
        substring = row[method]["answer_contains_gold"]
        if manually_correct != substring:
            disagreements.append({"id": row["id"], "method": method, "question": row["question"],
                                  "gold": row["gold_answer"], "answer": row[method]["answer"],
                                  "substring": substring, "manual": review[f"{method}_correctness"],
                                  "why": review["rationale"]})
display(pd.DataFrame(disagreements))
print("Metric disagreements:", len(disagreements))

,id,method,question,gold,answer,substring,manual,why
0,5ac55ea55542993e66e82377,naive,Pacific Mozart Ensemble performed which German...,Kurt Julian Weill,Kurt Weill,False,correct,Both correctly answer Kurt Weill from explicit...
1,5ac55ea55542993e66e82377,crag,Pacific Mozart Ensemble performed which German...,Kurt Julian Weill,Kurt Weill,False,correct,Both correctly answer Kurt Weill from explicit...
2,5ac15de55542991316484afb,naive,Were the films Crazy Love and Welcome to Nolly...,no,I don't have enough information in the provide...,True,abstain,The retrieved items are unrelated to the two f...
3,5ac15de55542991316484afb,crag,Were the films Crazy Love and Welcome to Nolly...,no,I don't have enough information in the provide...,True,abstain,The retrieved items are unrelated to the two f...
4,5a81537355429926c1cdad20,naive,What conference champions were putted against ...,Eastern Conference champion Orlando Magic agai...,The first NBA Finals in which the Orlando Magi...,False,correct,Naive RAG gives the exact 1995 conference cham...
5,5abe6f8455429965af743f03,naive,What country of origin does Robert brothers an...,French,France,False,correct,"Both answer France, which is semantically equi..."
6,5abe6f8455429965af743f03,crag,What country of origin does Robert brothers an...,French,France,False,correct,"Both answer France, which is semantically equi..."
7,5a72688c5542997f827839b2,naive,The Atik Valide Mosque and Valens Aqueduct are...,Turkey,"The Atik Valide Mosque is located in Istanbul,...",True,partially_correct,"Both name Turkey, but only the Atik Valide Mos..."
8,5adea0c5554299728e26c776,naive,Are both Helen Dunmore and M. P. Shiel of West...,no,I don't have enough information in the provide...,True,abstain,"Helen Dunmore is described as British, but M. ..."
9,5adea0c5554299728e26c776,crag,Are both Helen Dunmore and M. P. Shiel of West...,no,I don't have enough information in the provide...,True,abstain,"Helen Dunmore is described as British, but M. ..."


Metric disagreements: 10


## Exercise

Pick one `naive_better`, one `crag_better`, and one inadequate tie. For each, trace retrieval → supplied evidence → answer. Decide whether retrieval, CRAG evaluation, refinement, or generation caused the result. The next cell provides a compact scaffold from the completed human audit.

### Build a three-case error-analysis scaffold

**Motivation.** Practice causal diagnosis alongside win counts.

**Paper mapping.** CRAG's components create identifiable failure boundaries after shared retrieval.

**Next cell.** Select one reviewed example from each requested preference category and expose its evidence path.

**Failure signals.** A missing category is reported honestly; the notebook does not substitute an unrelated example.

**Read the output.** Use the shown action, titles, strips, and rationale to explain where the methods diverged.

In [11]:
for preference in ("naive_better", "crag_better", "tie_both_inadequate"):
    chosen = next((review for review in reviews if review["preference"] == preference), None)
    print("\n", preference.upper())
    if not chosen:
        print("No reviewed row in this category.")
        continue
    row = next(item for item in rows if item["id"] == chosen["id"])
    print("Q:", row["question"])
    print("Retrieved:", [hit["title"] for hit in row["hits"]])
    print("CRAG action/kept:", row["crag"]["action"], row["crag"]["kept_titles"])
    print("Diagnosis:", chosen["error_types"], "—", chosen["rationale"])


 NAIVE_BETTER
Q: After his curacy at the village that is a suburb of Scunthorpe, who was Industrial Chaplain to the Bishop of Lincoln?
Retrieved: ['William Everingham', 'Ernest Holmes (priest)', 'Bill Dudman']
CRAG action/kept: Correct ['Bill Dudman']
Diagnosis: ['generation_error'] — The full Bill Dudman passage directly supports the naive answer; CRAG retains the decisive chaplain sentence but still abstains.

 CRAG_BETTER
Q: What is the first two words of the fifth studio album of Joseph Edgar Foreman?
Retrieved: ['The Hungry Hustlerz: Starvation Is Motivation', 'Como Ama una Mujer', 'Too Much Stereo']
CRAG action/kept: Correct ['The Hungry Hustlerz: Starvation Is Motivation']
Diagnosis: ['generation_error'] — The album title is explicit; CRAG returns its first two words while naive RAG abstains despite receiving the same decisive passage.

 TIE_BOTH_INADEQUATE
Q: What nationality was Oliver Reed's character in the film Royal Flash?
Retrieved: ['Royal Flash (film)', 'Royal Flash', 

### Release Qdrant before the final report

**Motivation.** Avoid a Windows file lock after all retrieval-dependent work is complete.

**Paper mapping.** Operational cleanup is outside the comparison algorithm.

**Next cell.** Close the shared embedded client; the final reporting cell uses in-memory rows only.

**Failure signals.** If another kernel owns the store, shut it down. Do not delete lock files.

**Read the output.** The confirmation means this kernel released its client.

In [12]:
close_qdrant_client()
print("Qdrant client closed.")

Qdrant client closed.


### Final answer-by-answer comparison

**Motivation.** End with the complete human-readable evidence needed to inspect every verdict.

**Paper mapping.** This reporting layer compares the naive baseline with the paper-inspired corrective path.

**Next cell.** Print all 50 questions with gold, naive, CRAG, manual verdict, and rationale in slice order.

**Failure signals.** A NOT REVIEWED verdict means the answer/evidence fingerprint changed and requires a new human judgement.

**Read the output.** Read each block as a paired case; aggregate claims must agree with these 50 underlying decisions.

In [13]:
for number, row in enumerate(rows, 1):
    review = review_by_id.get(row["id"])
    verdict = review["preference"] if review else "NOT REVIEWED"
    reason = review["rationale"] if review else "The saved manual review is missing or stale for this exact output."
    print("=" * 100)
    print(f"Q{number:02d} [{row['id']}]: {row['question']}")
    print("Gold answer:", row["gold_answer"])
    print("Naive RAG:", row["naive"]["answer"])
    print("CRAG:", row["crag"]["answer"])
    print("Manual verdict:", verdict)
    print("Why:", reason)

Q01 [5add1d575542992c1e3a2540]: What nationality was Oliver Reed's character in the film Royal Flash?
Gold answer: Prussian
Naive RAG: I don't have enough information in the provided evidence to answer this question.
CRAG: I don't have enough information in the provided evidence to answer this question.
Manual verdict: tie_both_inadequate
Why: The passages identify Oliver Reed as Bismarck but never state that the character was Prussian, so neither method answers and both abstentions are evidence-aware.
Q02 [5ac55ea55542993e66e82377]: Pacific Mozart Ensemble performed which German composer's Der Lindberghflug in 2002?
Gold answer: Kurt Julian Weill
Naive RAG: Kurt Weill
CRAG: Kurt Weill
Manual verdict: tie_both_good
Why: Both correctly answer Kurt Weill from explicit evidence; the longer reference name Kurt Julian Weill makes literal substring matching undercount this valid alias.
Q03 [5ac3a76e554299741d48a2be]: Who released the song "With or Without You" first, Jai McDowall or U2?
Gold